# Đồ Án Machine Learning: Phân Loại Biển Báo Giao Thông
## Notebook 02: Kỹ Thuật Tiền Xử Lý & Tăng Cường Dữ Liệu (Data Preprocessing & Augmentation)

---
### Mục tiêu thực nghiệm:
1. **Khảo sát đặc thù bài toán ảnh giao thông:** Ảnh biển báo chụp từ camera hành trình ngoài thực tế thường xuyên chịu tác động của điều kiện thời tiết (nắng gắt, ngược sáng, trời âm u), góc nhìn nghiêng (perspective) do vị trí lắp đặt camera, và độ rung nhòe (motion blur) khi xe di chuyển tốc độ cao.
2. **Thiết kế & trực quan hóa các phép biến đổi ảnh (Data Augmentation):**
   - Xoay ngẫu nhiên (*Random Rotation*) trong ngưỡng an toàn góc nghiêng.
   - Biến dạng phối cảnh & co giãn (*Random Affine / Perspective*) mô phỏng góc nhìn 3D.
   - Thay đổi độ sáng, độ tương phản và bão hòa (*Color Jitter*) mô phỏng các thời điểm trong ngày.
   - Chuẩn hóa phân phối điểm ảnh (*Normalization*) tương thích với chuẩn ImageNet.
3. **So sánh trực quan Before vs After:** Đánh giá chất lượng ảnh sau khi đi qua pipeline tăng cường dữ liệu.
4. **Kiểm định với PyTorch Dataset & DataLoader:** Đảm bảo nạp dữ liệu tốc độ cao, không gây thắt cổ chai (bottleneck) trong quá trình huấn luyện.

In [ ]:
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance, ImageFilter
import torch
from torchvision import transforms

# Thêm thư mục gốc vào path để import các module src
project_root = Path('..').resolve()
if not (project_root / 'data').exists():
    project_root = Path('.').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.transforms import get_train_transforms, get_val_transforms
from src.data.dataset import TrafficSignDataset

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
print(f"[+] Đã import thành công các thư viện và module tiền xử lý từ {project_root}!")

### 1. Nạp Danh Mục Biển Báo và Ảnh Mẫu Thử Nghiệm
Chúng ta sẽ chọn ra một số mẫu biển báo đại diện thuộc các nhóm biển khác nhau (Biển cấm, Biển nguy hiểm, Biển hiệu lệnh, Biển chỉ dẫn) để tiến hành thử nghiệm các phép biến đổi.

In [ ]:
data_dir = project_root / 'data'
labels_path = data_dir / 'class_labels.csv'
templates_dir = data_dir / 'templates'

df_labels = pd.read_csv(labels_path)
print(f"Tổng số lớp biển báo quy chuẩn: {len(df_labels)}")

# Chọn 4 biển báo đại diện thuộc các nhóm khác nhau
sample_cids = [0, 128, 154, 71]  # P.101 (Đường cấm), R.301e (Hướng đi), W.201a (Chỗ ngoặt nguy hiểm), I.407a (Đường một chiều)
sample_images = {}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for idx, cid in enumerate(sample_cids):
    row = df_labels[df_labels['class_id'] == cid].iloc[0]
    img_path = list(templates_dir.glob(f"{cid}_*.png"))[0]
    img = Image.open(img_path).convert('RGB')
    sample_images[cid] = img
    
    axes[idx].imshow(img)
    axes[idx].set_title(f"Class {cid}: {row['sign_code']}\n{row['sign_name_vi']}", fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle("4 Mẫu Biển Báo Đại Diện Dùng Thử Nghiệm Augmentation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2. Thử Nghiệm Từng Phép Biến Đổi Đơn Lẻ (Individual Augmentations)

#### a) Phép Xoay Ngẫu Nhiên (Random Rotation)
* Trong điều kiện xe đi qua đường dốc, góc nghiêng của camera hành trình có thể bị lệch $\pm 10^\circ$ đến $\pm 20^\circ$.
* **Lưu ý quan trọng:** Không xoay góc quá lớn ($>30^\circ$) để tránh làm biến dạng hoặc đảo lộn ý nghĩa của các biển báo có hướng mũi tên chỉ đường (ví dụ R.301).

In [ ]:
base_img = sample_images[sample_cids[0]]  # P.101
angles = [-20, -10, 0, 10, 20]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, angle in enumerate(angles):
    rot_img = base_img.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=(128, 128, 128))
    axes[i].imshow(rot_img)
    axes[i].set_title(f"Góc xoay: {angle}°", fontsize=11)
    axes[i].axis('off')

plt.suptitle("Thử Nghiệm Phép Xoay Ảnh (Random Rotation với neutral background fill=128)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### b) Biến Dạng Phối Cảnh & Co Giãn Hình Học (Random Perspective / Affine)
Mô phỏng góc chụp của camera không vuông góc trực diện với mặt biển báo (biển báo nằm ở góc lề đường bên phải hoặc treo trên cao).

In [ ]:
perspective_transforms = [
    transforms.RandomPerspective(distortion_scale=0.15, p=1.0, fill=128),
    transforms.RandomPerspective(distortion_scale=0.25, p=1.0, fill=128),
    transforms.RandomPerspective(distortion_scale=0.35, p=1.0, fill=128),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), fill=128)
]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(sample_images[sample_cids[1]])
axes[0].set_title("Ảnh Gốc", fontsize=11, fontweight='bold')
axes[0].axis('off')

for i, t in enumerate(perspective_transforms):
    aug = t(sample_images[sample_cids[1]])
    axes[i+1].imshow(aug)
    axes[i+1].set_title(f"Biến thể Phối cảnh {i+1}", fontsize=11)
    axes[i+1].axis('off')

plt.suptitle("Thử Nghiệm Biến Dạng Phối Cảnh 3D & Dịch Chuyển (Perspective / Affine)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### c) Thay Đổi Độ Sáng & Độ Tương Phản (Color Jitter)
Mô phỏng sự biến thiên ánh sáng môi trường:
- Ánh sáng gắt ban trưa (High brightness).
- Ánh sáng chiều tà / bóng râm tán cây (Low brightness, low contrast).

In [ ]:
jitter_transforms = [
    transforms.ColorJitter(brightness=(0.6, 0.6)),   # Giảm sáng
    transforms.ColorJitter(brightness=(1.4, 1.4)),   # Tăng sáng
    transforms.ColorJitter(contrast=(0.6, 0.6)),     # Giảm tương phản
    transforms.ColorJitter(contrast=(1.4, 1.4))      # Tăng tương phản
]
titles = ["Giảm sáng (-40%)", "Tăng sáng (+40%)", "Giảm tương phản (-40%)", "Tăng tương phản (+40%)"]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(sample_images[sample_cids[2]])
axes[0].set_title("Ảnh Gốc", fontsize=11, fontweight='bold')
axes[0].axis('off')

for i, (t, title) in enumerate(zip(jitter_transforms, titles)):
    aug = t(sample_images[sample_cids[2]])
    axes[i+1].imshow(aug)
    axes[i+1].set_title(title, fontsize=11)
    axes[i+1].axis('off')

plt.suptitle("Thử Nghiệm Điều Chỉnh Độ Sáng & Độ Tương Phản (Color Jitter)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3. Pipeline Augmentation Toàn Diện Của Dự Án
Trong mã nguồn dự án tại `src/data/transforms.py`, chúng ta áp dụng pipeline hoàn chỉnh kết hợp:
1. `Resize((64, 64))` chuẩn hóa kích thước đầu vào đồng nhất.
2. `RandomRotation(degrees=20, fill=128)` xoay nhẹ với màu nền xám trung tính.
3. `RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08), fill=128)` mô phỏng khoảng cách xa gần và góc nhìn.
4. `ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25)` làm phong phú phổ màu sắc.
5. `ToTensor()` chuyển đổi sang Tensor dải giá trị $[0, 1]$.
6. `Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])` chuẩn hóa phân phối để mạng hội tụ nhanh nhất.

> **Đặc biệt lưu ý:** Tuyệt đối **KHÔNG** sử dụng `RandomHorizontalFlip` (lật ngang). Ví dụ: Biển *Cấm rẽ trái (P.123a)* nếu lật ngang sẽ trở thành *Cấm rẽ phải (P.123b)*, dẫn tới gán sai nhãn nghiêm trọng!

In [ ]:
train_transform = get_train_transforms(img_size=(64, 64))
val_transform = get_val_transforms(img_size=(64, 64))

# Hàm un-normalize để hiển thị ảnh từ Tensor
def unnormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.permute(1, 2, 0).numpy()
    img = np.clip(img * std + mean, 0, 1)
    return img

test_img = sample_images[sample_cids[3]]  # I.407a

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(test_img.resize((64, 64)))
axes[0].set_title("Ảnh Gốc (64x64)", fontsize=10, fontweight='bold')
axes[0].axis('off')

for i in range(1, 6):
    aug_tensor = train_transform(test_img)
    aug_np = unnormalize(aug_tensor)
    axes[i].imshow(aug_np)
    axes[i].set_title(f"Augmented #{i}", fontsize=10)
    axes[i].axis('off')

plt.suptitle("Mô Phỏng 5 Biến Thể Sinh Ra Từ Train Transforms Pipeline Hoàn Chỉnh", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4. Kiểm Thử Với PyTorch `TrafficSignDataset` & `DataLoader`
Nạp tập dữ liệu thực tế đã tiền xử lý để kiểm tra tính toàn vẹn của dữ liệu và nhãn tương ứng.

In [ ]:
summary_csv = data_dir / 'processed' / 'dataset_summary.csv'

if summary_csv.exists():
    dataset_train = TrafficSignDataset(summary_csv, base_dir=data_dir, split='train', transform=train_transform)
    dataset_val = TrafficSignDataset(summary_csv, base_dir=data_dir, split='val', transform=val_transform)
    
    print(f"[+] Số mẫu tập Huấn luyện (Train): {len(dataset_train)}")
    print(f"[+] Số mẫu tập Xác thực (Validation): {len(dataset_val)}")
    
    # Khởi tạo DataLoader
    train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=8, shuffle=True)
    images_batch, labels_batch = next(iter(train_loader))
    
    fig, axes = plt.subplots(1, 8, figsize=(18, 3))
    for i in range(8):
        cid = labels_batch[i].item()
        row = df_labels[df_labels['class_id'] == cid].iloc[0]
        axes[i].imshow(unnormalize(images_batch[i]))
        axes[i].set_title(f"ID {cid}\n{row['sign_code']}", fontsize=9)
        axes[i].axis('off')
    
    plt.suptitle("Một Batch 8 Mẫu Sau Khi Qua DataLoader & Train Transforms", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f"[!] Chưa tìm thấy file {summary_csv}. Hãy chạy script sinh dữ liệu trước!")

### 5. Kết Luận & Đánh Giá
1. **Hiệu quả của Data Augmentation:** Bằng cách áp dụng kết hợp xoay nhẹ, dịch chuyển góc nhìn, và thay đổi độ sáng, tập dữ liệu được mở rộng đa dạng, giúp mạng học được các đặc trưng bất biến với góc nghiêng và thời tiết.
2. **Quy tắc bảo toàn nhãn (Label Preserving):** Đã loại bỏ phép lật ngang/dọc để bảo vệ ngữ nghĩa của các biển báo định hướng mũi tên.
3. **Sẵn sàng huấn luyện:** Dữ liệu sau khi chuẩn hóa qua `TrafficSignDataset` có kích thước đồng nhất $3 	imes 64 	imes 64$, phân phối chuẩn $(0, 1)$, sẵn sàng làm đầu vào cho mô hình Baseline ở Notebook 03.